In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gridstatus

In [103]:
# Reading AI Datacenter locations, filtering out to get active datacenters
datacenters = pd.read_csv('ai_data_centers_consumption_and_locations.csv')

datacenters = datacenters[(datacenters['Country']=='United States') & (datacenters['Current power (MW)'] > 0)]
datacenters['State'] = datacenters['County'].str.split(', ').str[1]
datacenters['County'] = datacenters['County'].str.split(',').str[0]
datacenters.drop(columns = ['Current H100 equivalents',
                            'Current total capital cost (2025 USD billions)',
                            'Owner', 'Users', 'Selected Sources', 'Calculations sheet',
                            'Project', 'Current chip types', 'All chip types', 'Investors',
                            'Construction companies', 'Energy companies', 'Country','Address'], inplace = True)

datacenters_by_county = datacenters.groupby(['County','State']).agg(
    num_datacenters = ('Current power (MW)', 'count'),
    total_power_consumption_mw = ('Current power (MW)', 'sum')
    )
datacenters_by_county.sort_values(by=['num_datacenters','total_power_consumption_mw'], ascending = [False, False], inplace = True)
datacenters_by_county = datacenters_by_county.reset_index()
datacenters_by_county['County'] = datacenters_by_county['County'].str.replace(' County', '').str.strip()
datacenters_by_county

,County,State,num_datacenters,total_power_consumption_mw
0,Licking,OH,3,1108.00000
1,Henrico,VA,3,562.00000
2,Maricopa,AZ,3,449.00000
3,Bexar,TX,3,154.00000
4,Shelby,TN,2,1286.00000
5,Madison,MS,2,512.00000
6,Prince William,VA,2,323.00000
7,Sarpy,NE,2,317.00000
8,Ellis,TX,2,180.00000
9,Niagara,NY,2,100.00000


In [105]:
# reading service territory and utility data
df_st = pd.read_excel('EIA861_2024/Service_Territory_2024.xlsx', sheet_name='Counties_States')
df_ud = pd.read_excel('EIA861_2024/Utility_Data_2024.xlsx', sheet_name='States', header=1)

# filtering for grid columns and fixing the duplicate SPP column
iso_cols_raw = ['CAISO', 'ERCOT', 'PJM', 'MISO', 'SPP.1', 'ISONE', 'NYISO']
df_ud_clean = df_ud[['Utility Number', 'State'] + iso_cols_raw].rename(columns={'SPP.1': 'SPP'})

# merging grid data to counties
df_merged = df_st.merge(df_ud_clean, on=['Utility Number', 'State'], how='left')

# collapsing the 'Y' checkmarks into a single ISO column
isos = ['CAISO', 'ERCOT', 'PJM', 'MISO', 'SPP', 'ISONE', 'NYISO']
def get_iso(row):
    for iso in isos:
        if str(row[iso]).strip() == 'Y': return iso
    return 'BILATERAL'

df_merged['ISO'] = df_merged.apply(get_iso, axis=1)
county_to_grid = df_merged[['State', 'County', 'ISO']].drop_duplicates()

In [106]:
df_final = datacenters_by_county.merge(county_to_grid, on=['State', 'County'], how='left')
df_final

,County,State,num_datacenters,total_power_consumption_mw,ISO
0,Licking,OH,3,1108.00000,PJM
1,Licking,OH,3,1108.00000,BILATERAL
2,Henrico,VA,3,562.00000,PJM
3,Maricopa,AZ,3,449.00000,BILATERAL
4,Maricopa,AZ,3,449.00000,SPP
...,...,...,...,...,...
62,Cherokee,NC,1,65.00000,BILATERAL
63,Cumberland,NJ,1,50.00000,PJM
64,Whitfield,GA,1,28.00000,BILATERAL
65,Fulton,GA,1,17.32971,ERCOT
